# Formula 1 regulation PDF Q&A

This notebook builds a retrieval-augmented question-answering system over the PDFs in documents/. Choose an embedding backend (OpenAI, Hugging Face, or Ollama) and a vector store (Chroma or FAISS) in the configuration cell. The vector store keeps the regulation chunks and their embeddings locally; OpenAI chat generates the final cited answer.


## 1. Load libraries, API key, and backend choices

The embedding backend and vector store are selected independently. OpenAI embeddings use an API key; Hugging Face embeddings run locally; Ollama embeddings need a local Ollama server and a downloaded embedding model. Chroma and FAISS are both local vector stores.


In [1]:
from pathlib import Path

from dotenv import load_dotenv
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv(Path(".env"))

# Choose one: "openai", "huggingface", "ollama"
EMBEDDING_BACKEND = "huggingface"

# Choose one: "chroma", "faiss"
VECTOR_STORE_BACKEND = "faiss"


/var/folders/xw/sr9jsbqd6yqfw_bfrwthchy00000gn/T/ipykernel_24248/3206614735.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader


## 2. Find the PDFs

Put the PDFs you want to search in documents/. We keep each source filename so answers can cite where a passage came from.

In [2]:
pdf_dir = Path("documents")
pdf_paths = sorted(pdf_dir.glob("*.pdf"))

if not pdf_paths:
    raise FileNotFoundError(f"No PDF files found in {pdf_dir.resolve()}")

print(f"Found {len(pdf_paths)} PDFs:")
for path in pdf_paths:
    print("-", path.name)


Found 6 PDFs:
- FIA 2026 F1 Regulations - Section A [General Provisions] - Iss 03 - 2026-06-25.pdf
- FIA 2026 F1 Regulations - Section B [Sporting] - Iss 09 - 2026-10-01.pdf
- FIA 2026 F1 Regulations - Section C [Technical] - Iss 21 - 2026-10-19.pdf
- FIA 2026 F1 Regulations - Section D [Financial - F1 Teams] - Iss 08 - 2026-10-01.pdf
- FIA 2026 F1 Regulations - Section E [Financial – PU Manufacturers] - Iss 07 - 2026-10-19.pdf
- FIA 2026 F1 Regulations - Section F [Operational] - Iss 11 - 2026-10-01.pdf


## 3. Load each PDF page with LangChain

DirectoryLoader finds every PDF in the folder, and PyMuPDFLoader extracts its text page by page. Each result is a LangChain Document: its page_content holds the text, and metadata keeps the source PDF and zero-based page index.


In [3]:
loader = DirectoryLoader(
    path=str(pdf_dir),
    glob="*.pdf",
    loader_cls=PyMuPDFLoader,
)
docs = loader.load()

if not docs:
    raise ValueError("The PDFs were found, but no pages were loaded.")

print(f"Loaded {len(docs)} pages from {len(pdf_paths)} PDFs.")
print(docs[0].metadata)
print(docs[0].page_content[:500])


Loaded 613 pages from 6 PDFs.
{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-10-19T15:37:26+02:00', 'source': 'documents/FIA 2026 F1 Regulations - Section E [Financial – PU Manufacturers] - Iss 07 - 2026-10-19.pdf', 'file_path': 'documents/FIA 2026 F1 Regulations - Section E [Financial – PU Manufacturers] - Iss 07 - 2026-10-19.pdf', 'total_pages': 61, 'format': 'PDF 1.7', 'title': '', 'author': 'FIA', 'subject': '', 'keywords': '', 'moddate': '2026-10-19T15:37:26+02:00', 'trapped': '', 'modDate': "D:20261019153726+02'00'", 'creationDate': "D:20261019153726+02'00'", 'page': 0}
SECTION E: FINANCIAL REGULATIONS (POWER UNIT MANUFACTURERS) 
 
 
 
 
0  
E 
E1 
2026 Formula 1 Regulations – Section E [Financial - Power Unit Manufacturers] 
©2026 Fédération Internationale de l’Automobile 
19 October 2026
Issue 07
SECTION E: FINANCIAL REGULATIONS  
(POWER UNIT MANUFACTURERS) 
 
Version: 
 
 
Issue 07 
Status:  
 
 
PUBLISHE

## 4. Split pages into overlapping chunks with LangChain

RecursiveCharacterTextSplitter tries natural boundaries such as paragraphs and lines before falling back to smaller separators. split_documents() returns LangChain Documents and carries their source and page metadata into each chunk.


In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,  # characters
    chunk_overlap=200,  # characters shared by neighboring chunks
)

chunks = text_splitter.split_documents(docs)

if not chunks:
    raise ValueError("No text could be extracted. Scanned PDFs may need OCR.")

print(f"Created {len(chunks)} chunks.")
print(chunks[0].metadata)
print(chunks[0].page_content[:500])


Created 1813 chunks.
{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-10-19T15:37:26+02:00', 'source': 'documents/FIA 2026 F1 Regulations - Section E [Financial – PU Manufacturers] - Iss 07 - 2026-10-19.pdf', 'file_path': 'documents/FIA 2026 F1 Regulations - Section E [Financial – PU Manufacturers] - Iss 07 - 2026-10-19.pdf', 'total_pages': 61, 'format': 'PDF 1.7', 'title': '', 'author': 'FIA', 'subject': '', 'keywords': '', 'moddate': '2026-10-19T15:37:26+02:00', 'trapped': '', 'modDate': "D:20261019153726+02'00'", 'creationDate': "D:20261019153726+02'00'", 'page': 0}
SECTION E: FINANCIAL REGULATIONS (POWER UNIT MANUFACTURERS) 
 
 
 
 
0  
E 
E1 
2026 Formula 1 Regulations – Section E [Financial - Power Unit Manufacturers] 
©2026 Fédération Internationale de l’Automobile 
19 October 2026
Issue 07
SECTION E: FINANCIAL REGULATIONS  
(POWER UNIT MANUFACTURERS) 
 
Version: 
 
 
Issue 07 
Status:  
 
 
PUBLISHED 
Date: 

## 5. Select embeddings and build the vector store

Set `EMBEDDING_BACKEND` and `VECTOR_STORE_BACKEND` in the first code cell. This cell embeds the chunks and rebuilds the selected local index. OpenAI and Hugging Face model names, and the Ollama model name, can be changed in `create_embeddings()` below.


In [5]:
def create_embeddings(backend):
    if backend == "openai":
        print("Initializing OpenAI embeddings...")
        return OpenAIEmbeddings(
            model="text-embedding-3-small",
            show_progress_bar=True,
        )

    if backend == "huggingface":
        try:
            from langchain_huggingface import HuggingFaceEmbeddings
        except ImportError as error:
            raise ImportError(
                "Install Hugging Face support with: "
                "uv sync --extra huggingface"
            ) from error

        print("Initializing Hugging Face embeddings...")
        return HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2",
            show_progress=True,
        )    
    

    if backend == "ollama":
        try:
            from langchain_ollama import OllamaEmbeddings
        except ImportError as error:
            raise ImportError(
                "Install Ollama support with: uv sync --extra ollama"
            ) from error

        print("Initializing Ollama embeddings...")
        return OllamaEmbeddings(model="nomic-embed-text")

    raise ValueError(
        "EMBEDDING_BACKEND must be 'openai', 'huggingface', or 'ollama'."
    )


def create_vector_store(documents, embedding_function, backend, embedding_backend):
    if backend == "chroma":
        persist_directory = Path("chroma_db") / embedding_backend
        collection_name = "formula1_regulations"

        # Rebuild this collection so changed or removed PDFs do not leave stale chunks.
        existing_store = Chroma(
            collection_name=collection_name,
            embedding_function=embedding_function,
            persist_directory=str(persist_directory),
        )
        existing_store.delete_collection()

        return Chroma.from_documents(
            documents=documents,
            embedding=embedding_function,
            collection_name=collection_name,
            persist_directory=str(persist_directory),
        )

    if backend == "faiss":
        try:
            from langchain_community.vectorstores import FAISS
        except ImportError as error:
            raise ImportError(
                "Install FAISS support with: uv sync --extra faiss"
            ) from error

        vector_store = FAISS.from_documents(documents, embedding_function)
        index_directory = Path("faiss_index") / embedding_backend
        vector_store.save_local(str(index_directory))
        return vector_store

    raise ValueError("VECTOR_STORE_BACKEND must be 'chroma' or 'faiss'.")

embeddings = create_embeddings(EMBEDDING_BACKEND)
vector_store = create_vector_store(
    documents=chunks,
    embedding_function=embeddings,
    backend=VECTOR_STORE_BACKEND,
    embedding_backend=EMBEDDING_BACKEND,
)

if VECTOR_STORE_BACKEND == "chroma":
    stored_count = vector_store._collection.count()
else:
    stored_count = len(vector_store.index_to_docstore_id)

print(
    f"Generated embeddings for and indexed {stored_count} chunks using "
    f"{EMBEDDING_BACKEND} with {VECTOR_STORE_BACKEND}."
)


Initializing Hugging Face embeddings...


W1031 14:37:05.222000 24248 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/57 [00:00<?, ?it/s]

Generated embeddings for and indexed 1813 chunks using huggingface with faiss.


## 6. Retrieve passages for a question

The selected vector store embeds the question with the same embedding backend used for indexing and returns the most relevant stored chunks. Source and page metadata is retained for citations.


In [6]:
query = "What limits apply to aerodynamic testing?"
top_k = 5
retrieved = vector_store.similarity_search_with_relevance_scores(query, k=top_k)

for chunk, score in retrieved:
    source = Path(chunk.metadata["source"]).name
    page_number = int(chunk.metadata["page"]) + 1
    print(f"{score:.3f} | {source}, page {page_number}")
    print(chunk.page_content[:400], "\n")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

0.442 | FIA 2026 F1 Regulations - Section F [Operational] - Iss 11 - 2026-10-01.pdf, page 8
F4.1.3 
A three-dimensional representation of an F1 car or sub-component subject to Restricted 
Aerodynamic testing, defined either physically or digitally, will be considered for the purposes of 
this Article as a Restricted Aerodynamic Test Geometry (RATG) and save for where specifically 
permitted by this Article may not be added to, removed from, morphed or modified. In order to 
prevent Restr 

0.412 | FIA 2026 F1 Regulations - Section F [Operational] - Iss 11 - 2026-10-01.pdf, page 16
bodywork described by Articles C3.10 and C3.11 of the Technical Regulations respectively.  
b. 
A RATG is used which is more than 12 months old, or represents an FIA approved CAD geometry 
provided for this purpose and that no modification is made to this previously tested RATG or 
FIA approved geometry. 
During audit F1 Teams may be requested to demonstrate compliance of any such testing through  

0.390 | F

## 7. Generate a cited answer

The model receives only the retrieved passages. The prompt asks it to say when the evidence is missing and to cite claims with the PDF filename and page.

In [7]:
citation_options = sorted({
    (
        f"[{Path(chunk.metadata['source']).name}, "
        f"p. {int(chunk.metadata['page']) + 1}]"
    )
    for chunk, _score in retrieved
})
citation_list = "\n".join(f"- {citation}" for citation in citation_options)

context = "\n\n---\n\n".join(
    (
        f"Source: {Path(chunk.metadata['source']).name}, "
        f"page {int(chunk.metadata['page']) + 1}\n"
        f"{chunk.page_content}"
    )
    for chunk, _score in retrieved
)

prompt = (
    "Answer the question using only the source passages below.\n"
    "If the passages do not contain the answer, say you could not find it in the provided documents.\n"
    "Cite each factual claim by copying a citation exactly from the valid citation list.\n"
    "Never cite a placeholder such as filename.pdf or a source that is not in the list.\n\n"
    f"Question: {query}\n\n"
    f"Valid citations:\n{citation_list}\n\n"
    f"Source passages:\n{context}"
)

chat_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
response = chat_model.invoke(prompt)
print(response.content)


The limits that apply to aerodynamic testing include the following:

1. A three-dimensional representation of an F1 car or sub-component subject to Restricted Aerodynamic testing, known as a Restricted Aerodynamic Test Geometry (RATG), may not be modified except where specifically permitted. This includes not adding to, removing from, morphing, or modifying the RATG. Any attempt to derive aerodynamically independent results for subcomponents of a RATG is not permitted, nor is the use of boundary conditions to infer the effects of combining RATGs without accruing a further RATG ([FIA 2026 F1 Regulations - Section F [Operational] - Iss 11 - 2026-10-01.pdf, p. 8]).

2. Wind tunnel testing for the development of bodywork parts, even without aerodynamic force measurement, is classified as Restricted Aerodynamic testing. Additionally, any RATG used must be no more than 12 months old or represent an FIA approved CAD geometry, and no modifications can be made to previously tested RATGs or FIA 

## Try another question or backend

Change `query` and rerun steps 6 and 7 to ask another question. To change the embedding or vector store, edit the configuration in step 1 and rerun the notebook from the start so the chunks are re-indexed with that configuration. The Chroma index is saved under `chroma_db/<embedding_backend>/`; the FAISS index is saved under `faiss_index/<embedding_backend>/`.
